### Installation

In [1]:
%%capture
!pip install --upgrade uv
# 安裝純文字微調所需的最新套件
!uv pip install unsloth trl peft accelerator bitsandbytes xformers==0.0.32.post2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # 自動偵測硬體環境
load_in_4bit = True # 啟用 4-bit 量化，確保免費 Tesla T4 絕對不會爆視訊記憶體

# 載入角色扮演、語氣模仿能力極強的 Llama-3 文字模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 設置參數高效微調 (LoRA Adapters)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA 的 Rank 階數
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Unsloth 的記憶體優化關鍵
    random_state = 3407,
)
print("模型與 LoRA 配置完成！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

In [ ]:
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# 1. 使用 utf-8 讀取上傳的 cleandata.csv
# 假設你的推文內容在名為 'text' 或最後一欄（此處程式碼自動抓取最後一欄，如不對可修改 'text'）
df = pd.read_csv("cleandata.csv", encoding="utf-8")
tweet_column = df.columns[-1] # 自動抓取最後一欄，如果欄位名是 'text'，可改成 df['text']

raw_data = []
for tweet in df[tweet_column].dropna():
    tweet_str = str(tweet).strip()
    if tweet_str and tweet_str != "nan":
        # 建立對話結構
        raw_data.append({
            "conversations": [
                {"from": "human", "value": "What are your thoughts on this? Share your perspective."},
                {"from": "gpt", "value": tweet_str}
            ]
        })

# 2. 轉換為 Hugging Face Dataset 格式
dataset = Dataset.from_list(raw_data)

# 3. 套用 Llama-3 標準對話模板
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"},
)

def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"資料處理完成！共成功整理出 {len(dataset)} 筆馬斯克訓練數據。")

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 500筆資料建議先跑 60 步觀察 Loss，之後想更融入可調大（如 150）
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# 啟動訓練
trainer_stats = trainer.train()

In [ ]:
# 開啟推理模式
FastLanguageModel.for_inference(model)

# 測試問題
messages = [
    {"role": "user", "content": "What is the key to successfully engineering a rocket that can landing by itself?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 150, use_cache = True, temperature = 0.85)
print("\n=== AI 馬斯克的回應 ===")
print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1])

In [ ]:
model.save_pretrained("elon_musk_lora")
tokenizer.save_pretrained("elon_musk_lora")
print("模型權重已成功儲存至 elon_musk_lora 資料夾！")